In [1]:
from rapidfuzz import fuzz, process
from collections import defaultdict
import logging
import re

# -------------------------------
# Logging
# -------------------------------
logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

# -------------------------------
# Hàm tiện ích
# -------------------------------
def remove_prefix(name):
    """Loại bỏ tiền tố/suffix hành chính phổ biến"""
    name = re.sub(r"^(Xã|Phường|Thị trấn)\s+", "", name)
    name = re.sub(r"^(Huyện|Quận|Thị xã|Thành phố)\s+", "", name)
    name = re.sub(r"\s+(Huyện|Quận|Thị xã|Thành phố)$", "", name)
    return name.strip()

def _load_places_file(file_path):
    if not file_path:
        return []
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

# -------------------------------
# Fuzzy matching
# -------------------------------
def match_fuzzy_topk(span, candidate_list, top_k=5, scorer=fuzz.WRatio):
    if not candidate_list:
        return []
    results = process.extract(span, candidate_list, scorer=scorer, limit=top_k)
    return [(r[0], r[1]) for r in results]

def select_best_candidate(span, top_candidates):
    span_tokens = set(span.lower().strip().split())
    best_score = -1
    best_candidate = None
    for cand, score in top_candidates:
        cand_tokens = set(cand.lower().strip().split())
        token_overlap = len(span_tokens & cand_tokens)
        combined_score = score + 20 * token_overlap
        logger.info(f"Candidate: '{cand}', fuzzy score: {score}, token overlap: {token_overlap}, combined: {combined_score}")
        if combined_score > best_score:
            best_score = combined_score
            best_candidate = cand
    return best_candidate

# -------------------------------
# Build dict từ file
# -------------------------------
def build_mappings(file_path):
    ward_to_districts = defaultdict(list)  # ward gốc -> list of (district, province)
    district_to_provinces = defaultdict(set)  # district -> set(province)
    normalized_ward_map = defaultdict(list)
    normalized_district_map = defaultdict(list)
    normalized_province_map = defaultdict(list)

    with open(file_path, "r", encoding="utf-8") as f:
        next(f)  # bỏ header
        for line in f:
            line = line.strip()
            if not line:
                continue
            ward, district, province = line.split("\t")

            # ward -> list (district, province)
            ward_to_districts[ward].append((district, province))
            # district -> province
            district_to_provinces[district].add(province)

            # build normalized maps
            normalized_ward_map[remove_prefix(ward)].append(ward)
            normalized_district_map[remove_prefix(district)].append(district)
            normalized_province_map[remove_prefix(province)].append(province)

    # chuyển set sang list
    district_to_provinces = {d: list(provs) for d, provs in district_to_provinces.items()}

    return ward_to_districts, district_to_provinces, normalized_ward_map, normalized_district_map, normalized_province_map

# -------------------------------
# correct_address với normalized map
# -------------------------------
def correct_address(span, wards_list, districts_list, provinces_list,
                    ward_to_districts, district_to_provinces,
                    normalized_ward_map, normalized_district_map, normalized_province_map,
                    top_k=5, verbose=True):

    span_norm = remove_prefix(span)
    if verbose:
        logger.info(f"\nProcessing span: '{span}' -> normalized: '{span_norm}'")

    # 1️⃣ Match ward
    candidate_wards = normalized_ward_map.get(span_norm, wards_list)
    topk_wards = match_fuzzy_topk(span, candidate_wards, top_k=top_k)
    if verbose:
        # In kèm district/province tương ứng
        topk_info = [(w, s, ward_to_districts.get(w, [])) for w, s in topk_wards]
        logger.info(f"Top-K wards with districts/provinces: {topk_info}")

    ward = select_best_candidate(span, topk_wards) if topk_wards else None
    if ward:
        dp_list = ward_to_districts.get(ward, [])
        districts = [d for d, _ in dp_list]
        provinces = [p for _, p in dp_list]
        districts = list(set(districts))
        provinces = list(set(provinces))
        if verbose:
            logger.info(f"Selected ward: {ward}, districts: {districts}, provinces: {provinces}")
        return {'ward': ward, 'districts': districts, 'provinces': provinces}

    # 2️⃣ Match district
    candidate_districts = normalized_district_map.get(span_norm, districts_list)
    topk_districts = match_fuzzy_topk(span, candidate_districts, top_k=top_k)
    if verbose:
        topk_info = [(d, s, district_to_provinces.get(d, [])) for d, s in topk_districts]
        logger.info(f"Top-K districts with provinces: {topk_info}")

    district = select_best_candidate(span, topk_districts) if topk_districts else None
    if district:
        provinces = district_to_provinces.get(district, [])
        if verbose:
            logger.info(f"Selected district: {district}, provinces: {provinces}")
        return {'ward': None, 'districts': [district], 'provinces': provinces}

    # 3️⃣ Match province
    candidate_provinces = normalized_province_map.get(span_norm, provinces_list)
    topk_provinces = match_fuzzy_topk(span, candidate_provinces, top_k=top_k)
    province = select_best_candidate(span, topk_provinces) if topk_provinces else None
    if verbose:
        logger.info(f"Top-K provinces: {topk_provinces}")
        logger.info(f"Selected province: {province}")
    return {'ward': None, 'districts': [], 'provinces': [province] if province else []}

# -------------------------------
# Ví dụ test
# -------------------------------
if __name__ == "__main__":
    file_path = "/home/nampv1/projects/asr/asr-evaluation-app/test_samples/old_communes_10047_not_processed.txt"

    # Build dict
    ward_to_districts, district_to_provinces, normalized_ward_map, normalized_district_map, normalized_province_map = build_mappings(file_path)

    wards_list = list(ward_to_districts.keys())
    districts_list = list(district_to_provinces.keys())
    provinces_list = list({p for plist in district_to_provinces.values() for p in plist})

    test_spans = ["\Bưu gửi được chuyển đến: Ms. Huệ –, Công ty TNHH TM Ngọc Quê. Địa chỉ: Km82 Quốc lộ 1a, thôn Rừng Dông, xã Hữu Lũng, tỉnh Ninh Thuậtttt. Điện thoại liên hệ: 0985 378168."]

    test_spans = [
        "Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai, quận Ba Đình, Hà Nội"
    ]

    for span in test_spans:
        corrected = correct_address(span, wards_list, districts_list, provinces_list,
                                    ward_to_districts, district_to_provinces,
                                    normalized_ward_map, normalized_district_map, normalized_province_map,
                                    verbose=True)
        logger.info(f"Final corrected: {corrected}")



Processing span: 'Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai, quận Ba Đình, Hà Nội' -> normalized: 'Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai, quận Ba Đình, Hà Nội'
Top-K wards with districts/provinces: [('Phường Liễu Giai', 85.5, [('Quận Ba Đình', 'Thành phố Hà Nội')]), ('Phường Ngọc Hà', 85.5, [('Quận Ba Đình', 'Thành phố Hà Nội'), ('Thành phố Hà Giang', 'Tỉnh Hà Giang')]), ('Phường Hàng Mã', 85.5, [('Quận Hoàn Kiếm', 'Thành phố Hà Nội')]), ('Phường Hàng Buồm', 85.5, [('Quận Hoàn Kiếm', 'Thành phố Hà Nội')]), ('Phường Hàng Đào', 85.5, [('Quận Hoàn Kiếm', 'Thành phố Hà Nội')])]
Candidate: 'Phường Liễu Giai', fuzzy score: 85.5, token overlap: 1, combined: 105.5
Candidate: 'Phường Ngọc Hà', fuzzy score: 85.5, token overlap: 1, combined: 105.5
Candidate: 'Phường Hàng Mã', fuzzy score: 85.5, token overlap: 1, combined: 105.5
Candidate: 'Phường Hàng Buồm', fuzzy score: 85.5, token overlap: 1, combined: 105.

In [7]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

tokenizer = AutoTokenizer.from_pretrained("NlpHUST/ner-vietnamese-electra-base")
model = AutoModelForTokenClassification.from_pretrained("NlpHUST/ner-vietnamese-electra-base") 
nlp = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple") 
text = "Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai Ba Đình Hà Nội" 
ner_results = nlp(text)

/home/nampv1/anaconda3/envs/asr/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [8]:
for r in ner_results:
    span = r["word"]
    corrected = correct_address(
        span,
        wards_list, districts_list, provinces_list,
        ward_to_districts, district_to_provinces,
        normalized_ward_map, normalized_district_map, normalized_province_map,
        verbose=True
    )
    print(f"Span: {span} -> {corrected}")



Processing span: 'xã Lĩnh Toại' -> normalized: 'xã Lĩnh Toại'
Top-K wards with districts/provinces: [('Xã Lĩnh Toại', 91.66666666666666, [('Huyện Hà Trung', 'Tỉnh Thanh Hóa')]), ('Xã Vĩnh Lại', 69.56521739130434, [('Huyện Lâm Thao', 'Tỉnh Phú Thọ')]), ('Xã Vĩnh Đại', 69.56521739130434, [('Huyện Tân Hưng', 'Tỉnh Long An')]), ('Xã Vĩnh Tiến', 66.66666666666667, [('Huyện Kim Bôi', 'Tỉnh Hoà Bình'), ('Huyện Vĩnh Lộc', 'Tỉnh Thanh Hóa')]), ('Xã Vĩnh Thái', 66.66666666666667, [('Huyện Vĩnh Linh', 'Tỉnh Quảng Trị'), ('Thành phố Nha Trang', 'Tỉnh Khánh Hòa')])]
Candidate: 'Xã Lĩnh Toại', fuzzy score: 91.66666666666666, token overlap: 3, combined: 151.66666666666666
Candidate: 'Xã Vĩnh Lại', fuzzy score: 69.56521739130434, token overlap: 1, combined: 89.56521739130434
Candidate: 'Xã Vĩnh Đại', fuzzy score: 69.56521739130434, token overlap: 1, combined: 89.56521739130434
Candidate: 'Xã Vĩnh Tiến', fuzzy score: 66.66666666666667, token overlap: 1, combined: 86.66666666666667
Candidate: 'Xã Vĩnh 

Span: xã Lĩnh Toại -> {'ward': 'Xã Lĩnh Toại', 'districts': ['Huyện Hà Trung'], 'provinces': ['Tỉnh Thanh Hóa']}
Span: huyện Hà Trung -> {'ward': 'Phường Phương Liên – Trung Tự', 'districts': ['Quận Đống Đa'], 'provinces': ['Thành phố Hà Nội']}
Span: tỉnh Thanh Hóa -> {'ward': 'Phường Thanh Xuân Trung', 'districts': ['Quận Thanh Xuân'], 'provinces': ['Thành phố Hà Nội']}
Span: xã Liễu Giai -> {'ward': 'Phường Liễu Giai', 'districts': ['Quận Ba Đình'], 'provinces': ['Thành phố Hà Nội']}
Span: Ba Đình -> {'ward': 'Xã Ba Đình', 'districts': ['Huyện Nga Sơn'], 'provinces': ['Tỉnh Thanh Hóa']}
Span: Hà Nội -> {'ward': 'Phường Ngọc Hà', 'districts': ['Thành phố Hà Giang', 'Quận Ba Đình'], 'provinces': ['Thành phố Hà Nội', 'Tỉnh Hà Giang']}


In [31]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from rapidfuzz import fuzz
import re

# -----------------------------
# Load NER model
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained("NlpHUST/ner-vietnamese-electra-base")
model = AutoModelForTokenClassification.from_pretrained("NlpHUST/ner-vietnamese-electra-base")
nlp = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# -----------------------------
# Giả DB địa chỉ
# -----------------------------
db = {
    "wards": [
        ("Xã Lĩnh Toại", "Huyện Hà Trung", "Tỉnh Thanh Hóa"),
        ("Thị trấn Hà Trung", "Huyện Hà Trung", "Tỉnh Thanh Hóa"),
        ("Phường Liễu Giai", "Quận Ba Đình", "Thành phố Hà Nội"),
        ("Phường Ngọc Hà", "Quận Ba Đình", "Thành phố Hà Nội"),
    ],
    "districts": [
        ("Huyện Hà Trung", "Huyện Hà Trung", "Tỉnh Thanh Hóa"),
        ("Quận Ba Đình", "Quận Ba Đình", "Thành phố Hà Nội"),
    ],
    "provinces": [
        ("Tỉnh Thanh Hóa", "Huyện Hà Trung", "Tỉnh Thanh Hóa"),
        ("Thành phố Hà Nội", "Quận Ba Đình", "Thành phố Hà Nội"),
    ]
}

# -----------------------------
# Helper functions
# -----------------------------
def normalize_text(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip().lower())

def detect_level(span: str):
    s = normalize_text(span)
    if s.startswith("xã") or s.startswith("phường") or s.startswith("thị trấn"):
        return "ward"
    if s.startswith("quận") or s.startswith("huyện") or s.startswith("thành phố") or s.startswith("thị xã"):
        return "district"
    if s.startswith("tỉnh") or s.startswith("thành phố"):
        return "province"
    return "unknown"

def resolve_span_with_context(span, db, context):
    norm = normalize(span)

    # --- Check Province ---
    prov = match_best(norm, db["provinces"])
    if prov:
        context["province"] = prov
        context["district"] = None
        context["ward"] = None
        return {"span": span, "ward": None, "district": None, "province": prov}

    # --- Check District ---
    dist = match_best(norm, db["districts"])
    if dist:
        context["district"] = dist
        # gắn luôn province cha
        context["province"] = db["district_to_province"].get(dist, context["province"])
        return {"span": span, "ward": None, "district": dist, "province": context["province"]}

    # --- Check Ward ---
    ward = match_best(norm, db["wards"])
    if ward:
        context["ward"] = ward
        # gắn luôn district & province cha
        dist = db["ward_to_district"].get(ward)
        if dist:
            context["district"] = dist
            prov = db["district_to_province"].get(dist)
            if prov:
                context["province"] = prov
        return {"span": span, "ward": ward, "district": context["district"], "province": context["province"]}

    return {"span": span, "ward": None, "district": None, "province": None}


def merge_results(results):
    """Ghép các kết quả thành chuỗi địa chỉ chuẩn hóa"""
    wards = [r["ward"] for r in results if r["ward"]]
    districts = [r["district"] for r in results if r["district"]]
    provinces = [r["province"] for r in results if r["province"]]
    parts = []
    if wards: parts.append(wards[-1])  # lấy ward cuối cùng
    if districts: parts.append(districts[-1])
    if provinces: parts.append(provinces[-1])
    return ", ".join(parts)

# -----------------------------
# Test
# -----------------------------
text = "Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai Ba Đình Hà Nội, rồi đến Tam Sơn Cẩm Khê PHú Thọ"
print("text:", text)
ner_results = nlp(text)

spans = [r["word"] for r in ner_results]
context = {"wards": [], "districts": [], "provinces": []}
resolved = []

for span in spans:
    print(span)
    resolved.append(resolve_span_with_context(span, db, context))

print("=== Resolved spans ===")
for r in resolved:
    print(r)

print("\n=== Merged address ===")
print(merge_results(resolved))


Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


text: Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai Ba Đình Hà Nội, rồi đến Tam Sơn Cẩm Khê PHú Thọ
xã Lĩnh Toại
huyện Hà Trung
tỉnh Thanh Hóa
xã Liễu Giai
Ba Đình
Hà Nội
Tam Sơn
Cẩm Khê
PHú Thọ
=== Resolved spans ===
{'span': 'xã Lĩnh Toại', 'ward': None, 'district': None, 'province': None}
{'span': 'huyện Hà Trung', 'ward': None, 'district': None, 'province': None}
{'span': 'tỉnh Thanh Hóa', 'ward': None, 'district': None, 'province': None}
{'span': 'xã Liễu Giai', 'ward': None, 'district': None, 'province': None}
{'span': 'Ba Đình', 'ward': None, 'district': None, 'province': None}
{'span': 'Hà Nội', 'ward': None, 'district': None, 'province': None}
{'span': 'Tam Sơn', 'ward': None, 'district': None, 'province': None}
{'span': 'Cẩm Khê', 'ward': None, 'district': None, 'province': None}
{'span': 'PHú Thọ', 'ward': None, 'district': None, 'province': None}

=== Merged address ===



In [32]:
def merge_addresses(resolved):
    addresses = []
    current = {"ward": None, "district": None, "province": None}

    for r in resolved:
        if r["province"]:
            # Nếu province thay đổi => push địa chỉ trước
            if current["province"] and r["province"] != current["province"]:
                parts = [p for p in [current["ward"], current["district"], current["province"]] if p]
                if parts:
                    addresses.append(", ".join(parts))
                current = {"ward": None, "district": None, "province": None}

        # update current
        for key in ["ward", "district", "province"]:
            if r[key]:
                current[key] = r[key]

    # push cuối cùng
    parts = [p for p in [current["ward"], current["district"], current["province"]] if p]
    if parts:
        addresses.append(", ".join(parts))

    return addresses

# -----------------------------
# Test
# -----------------------------
text = "Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai Ba Đình Hà Nội, rồi đến xã Tam Sơn Cẩm Khê PHú Thọ"
print("text:", text)
ner_results = nlp(text)
print("ner_results:", ner_results)

spans = [r["word"] for r in ner_results]
context = {"wards": [], "districts": [], "provinces": []}
resolved = [resolve_span_with_context(span, db, context) for span in spans]

print("=== Resolved spans ===")
for r in resolved:
    print(r)

print("\n=== Merged addresses ===")
for addr in merge_addresses(resolved):
    print(addr)


text: Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai Ba Đình Hà Nội, rồi đến xã Tam Sơn Cẩm Khê PHú Thọ
ner_results: [{'entity_group': 'LOCATION', 'score': 0.9990113, 'word': 'xã Lĩnh Toại', 'start': 23, 'end': 35}, {'entity_group': 'LOCATION', 'score': 0.9990899, 'word': 'huyện Hà Trung', 'start': 37, 'end': 51}, {'entity_group': 'LOCATION', 'score': 0.99913716, 'word': 'tỉnh Thanh Hóa', 'start': 53, 'end': 67}, {'entity_group': 'LOCATION', 'score': 0.9990354, 'word': 'xã Liễu Giai', 'start': 71, 'end': 83}, {'entity_group': 'LOCATION', 'score': 0.9990537, 'word': 'Ba Đình', 'start': 84, 'end': 91}, {'entity_group': 'LOCATION', 'score': 0.8743003, 'word': 'Hà Nội', 'start': 92, 'end': 98}, {'entity_group': 'LOCATION', 'score': 0.9990292, 'word': 'xã Tam Sơn', 'start': 108, 'end': 118}, {'entity_group': 'LOCATION', 'score': 0.99697614, 'word': 'Cẩm Khê', 'start': 119, 'end': 126}, {'entity_group': 'LOCATION', 'score': 0.99738485, 'word': 'PHú Thọ', 

In [33]:
# Run this cell after you've loaded the NER pipeline and the resolver functions
# (build_mappings, resolve_spans, merge_addresses) defined previously.

text = "Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai Ba Đình Hà Nội, rồi đến xã Tam Sơn Cẩm Khê PHú Thọ"
print("text:", text)

# run NER (you already created `nlp` previously)
ner_results = nlp(text)
print("\nNER results:")
for r in ner_results:
    print(r)

# extract spans in order
spans = [ent["word"] for ent in ner_results if ent.get("entity_group", "").upper() in ("LOC", "LOCATION", "LOCATION") or True]
# note: using ent["word"] directly since your NER returns LOCATION only; adjust filter if needed

# load mappings (replace path if different)
file_path = "/home/nampv1/projects/asr/asr-evaluation-app/test_samples/old_communes_10047_not_processed.txt"
mappings = build_mappings(file_path)

# resolve spans with verbose logs so you can see decisions
resolved = resolve_spans(spans, mappings, FUZZY_PART_THRESH=55, FUZZY_FULL_THRESH=72, verbose=True)

print("\n=== Resolved spans ===")
for r in resolved:
    print(r)

print("\n=== Merged addresses ===")
addrs = merge_addresses(resolved)
for a in addrs:
    print(a)


text: Hàng hóa vận chuyển từ xã Lĩnh Toại, huyện Hà Trung, tỉnh Thanh Hóa ra xã Liễu Giai Ba Đình Hà Nội, rồi đến xã Tam Sơn Cẩm Khê PHú Thọ

NER results:
{'entity_group': 'LOCATION', 'score': 0.9990113, 'word': 'xã Lĩnh Toại', 'start': 23, 'end': 35}
{'entity_group': 'LOCATION', 'score': 0.9990899, 'word': 'huyện Hà Trung', 'start': 37, 'end': 51}
{'entity_group': 'LOCATION', 'score': 0.99913716, 'word': 'tỉnh Thanh Hóa', 'start': 53, 'end': 67}
{'entity_group': 'LOCATION', 'score': 0.9990354, 'word': 'xã Liễu Giai', 'start': 71, 'end': 83}
{'entity_group': 'LOCATION', 'score': 0.9990537, 'word': 'Ba Đình', 'start': 84, 'end': 91}
{'entity_group': 'LOCATION', 'score': 0.8743003, 'word': 'Hà Nội', 'start': 92, 'end': 98}
{'entity_group': 'LOCATION', 'score': 0.9990292, 'word': 'xã Tam Sơn', 'start': 108, 'end': 118}
{'entity_group': 'LOCATION', 'score': 0.99697614, 'word': 'Cẩm Khê', 'start': 119, 'end': 126}
{'entity_group': 'LOCATION', 'score': 0.99738485, 'word': 'PHú Thọ', 'start':

TypeError: resolve_spans() got an unexpected keyword argument 'FUZZY_PART_THRESH'